# AEMO WEM Demand Data Fetch and Analysis

In [10]:
from datetime import date, timedelta
import pandas as pd
import httpx

## 1. Fetching Data from AEMO WEM Data Portal

This section fetches the daily Operational Demand and Withdrawal data from the AEMO WEM (Wholesale Electricity Market) data portal. The data is available as JSON.

In [11]:
day = date.today() - timedelta(days=2)
url = (
    "https://data.wa.aemo.com.au/public/market-data/wemde/"
    f"operationalDemandWithdrawal/dailyFiles/OperationalDemandAndWithdrawal_{day.isoformat()}.json"
)


async def fetch_wem_demand_data(url):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url)
    resp.raise_for_status()
    payload = resp.json()
    print("url:", url)
    print("status:", resp.status_code)
    print("top-level keys:", list(payload.keys()))

    records = payload.get("data", {}).get("data", [])
    print(f"\n{len(records)} demand records; first one:")
    print(records[0] if records else None)
    return records

## 2. Processing and Loading Data into DataFrame

This code extracts the demand records from the fetched JSON and loads them into a pandas DataFrame.

In [12]:
try:
    wem_records = await fetch_wem_demand_data(url)

    if wem_records:
        df_wem = pd.DataFrame(wem_records)
        print("\nData loaded into pandas DataFrame for AEMO WEM. First 5 rows:")
        display(df_wem.head())
    else:
        print("No data records found to create a DataFrame.")
except Exception as exc:
    print(f"aemo-wem fetch failed ({type(exc).__name__}): {exc}")

url: https://data.wa.aemo.com.au/public/market-data/wemde/operationalDemandWithdrawal/dailyFiles/OperationalDemandAndWithdrawal_2026-08-03.json
status: 200
top-level keys: ['data', 'errors', 'warnings', 'infos', 'transactionId']

288 demand records; first one:
{'dispatchInterval': '2026-08-03T08:00:00+08:00', 'asAtTimeStamp': '2026-08-03T08:05:00+08:00', 'operationalDemand': 2811.36865, 'operationalWithdrawal': -7.22192}

Data loaded into pandas DataFrame for AEMO WEM. First 5 rows:


,dispatchInterval,asAtTimeStamp,operationalDemand,operationalWithdrawal
0,2026-08-03T08:00:00+08:00,2026-08-03T08:05:00+08:00,2811.36865,-7.22192
1,2026-08-03T08:05:00+08:00,2026-08-03T08:10:00+08:00,2754.77300,-14.01500
2,2026-08-03T08:10:00+08:00,2026-08-03T08:15:00+08:00,2697.02832,-12.38281
3,2026-08-03T08:15:00+08:00,2026-08-03T08:20:00+08:00,2642.74658,-17.38183
4,2026-08-03T08:20:00+08:00,2026-08-03T08:25:00+08:00,2580.19458,-17.41958


In [13]:
# ## 3. Displaying DataFrame as Pretty JSON

if "df_wem" in locals() and not df_wem.empty:
    json_str_wem = df_wem.to_json(orient="records", indent=2)
    print(json_str_wem)
else:
    print("WEM DataFrame is empty or not created, cannot generate JSON.")

[
  {
    "dispatchInterval":"2026-08-03T08:00:00+08:00",
    "asAtTimeStamp":"2026-08-03T08:05:00+08:00",
    "operationalDemand":2811.36865,
    "operationalWithdrawal":-7.22192
  },
  {
    "dispatchInterval":"2026-08-03T08:05:00+08:00",
    "asAtTimeStamp":"2026-08-03T08:10:00+08:00",
    "operationalDemand":2754.773,
    "operationalWithdrawal":-14.015
  },
  {
    "dispatchInterval":"2026-08-03T08:10:00+08:00",
    "asAtTimeStamp":"2026-08-03T08:15:00+08:00",
    "operationalDemand":2697.02832,
    "operationalWithdrawal":-12.38281
  },
  {
    "dispatchInterval":"2026-08-03T08:15:00+08:00",
    "asAtTimeStamp":"2026-08-03T08:20:00+08:00",
    "operationalDemand":2642.74658,
    "operationalWithdrawal":-17.38183
  },
  {
    "dispatchInterval":"2026-08-03T08:20:00+08:00",
    "asAtTimeStamp":"2026-08-03T08:25:00+08:00",
    "operationalDemand":2580.19458,
    "operationalWithdrawal":-17.41958
  },
  {
    "dispatchInterval":"2026-08-03T08:25:00+08:00",
    "asAtTimeStamp":"2026-0

## 4. Explanation of AEMO WEM Demand Data Columns

Based on the fetched AEMO WEM data, here's what each column represents:

*   **dispatchInterval**: The start time of the 5-minute dispatch interval in ISO 8601 format (e.g., `2026-08-03T08:00:00+08:00`).
*   **asAtTimeStamp**: The timestamp when the data was published or made available for that interval.
*   **operationalDemand**: The actual operational demand (in MW) for the given dispatch interval.
*   **operationalWithdrawal**: The total operational withdrawal (in MW) for the given dispatch interval. This typically refers to the total energy consumed by loads connected to the system.